In [6]:
import requests
import pandas as pd

url = "https://api.jamendo.com/v3.0/tracks/"
params = {
    "client_id": "31ffae09",
    "format": "json",
    "limit": 200,
    "include": "musicinfo",
}

response = requests.get(url, params=params)
data = response.json()
print(data["headers"])

{'status': 'success', 'code': 0, 'error_message': '', 'warnings': '', 'results_count': 200, 'next': 'https://api.jamendo.com/v3.0/tracks?client_id=31ffae09&format=json&limit=200&include=musicinfo&offset=200'}


In [7]:
all_tags = []

for track in data["results"]:
    vartags = track["musicinfo"]["tags"]["vartags"]
    all_tags.extend(vartags)

print("Total mood tags:", len(all_tags))
print(pd.Series(all_tags).value_counts().head(20))

Total mood tags: 59
neutral      17
peaceful     12
acappella     9
chorale       5
undefined     3
engage        3
spiritual     2
sad           2
action        1
energetic     1
emotional     1
happy         1
religious     1
enfants       1
Name: count, dtype: int64


In [11]:
import time

all_tracks = []

for offset in range(0, 1000, 200):
    params = {
        "client_id": "31ffae09",
        "format": "json",
        "limit": 200,
        "offset": offset,
        "include": "musicinfo",
        "order": "popularity_total",
        "datebetween": "2021-01-01_2025-12-31",
    }

    for attempt in range(3):
        response = requests.get(url, params=params)
        results = response.json()["results"]
        if len(results) > 0:
            break
        time.sleep(2)

    all_tracks.extend(results)
    print("offset", offset, "->", len(results), "tracks")
    time.sleep(1)

print("Total:", len(all_tracks))

offset 0 -> 200 tracks
offset 200 -> 200 tracks
offset 400 -> 200 tracks
offset 600 -> 200 tracks
offset 800 -> 200 tracks
Total: 1000


In [13]:
rows = []

for track in all_tracks:
    info = track["musicinfo"]
    tags = info["tags"]
    rows.append({
        "track_id": track["id"],
        "track_name": track["name"],
        "artist": track["artist_name"],
        "release_date": track["releasedate"],
        "duration_sec": track["duration"],
        "speed": info["speed"],
        "vocal_instrumental": info["vocalinstrumental"],
        "genres": ", ".join(tags["genres"]),
        "mood": tags["vartags"][0] if tags["vartags"] else None,
    })

jamendo_df = pd.DataFrame(rows)
jamendo_df.to_csv("jamendo_tracks.csv", index=False)
print(jamendo_df.shape)
jamendo_df.head()

(1000, 9)


,track_id,track_name,artist,release_date,duration_sec,speed,vocal_instrumental,genres,mood
0,1932670,RED LIGHT,Egor Budennyy,2022-03-17,190,medium,vocal,"pop, disco",groovy
1,2034080,Music Soothe - Meditation Music,StimiBeats,2023-06-01,216,low,instrumental,"newage, ambient",relaxing
2,1886257,Alone,Color Out,2021-09-20,245,medium,vocal,"emo, poprock",emotional
3,1876086,Love of My Life,Ben Lvcas,2021-08-04,161,high,vocal,"edm, pop, electronic",energetic
4,1826189,Not the One to Say (I Told You So) (24-bit Mas...,Cole Powell,2021-01-23,235,medium,vocal,"pop, jazz, singersongwriter",love
